In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="prompt-engineering",
    notebook_name="04_prompt_evaluation.ipynb"
)

# Prompt Evaluation — Hands-On

In this notebook, you will build an evaluation pipeline for prompts:
test suites, metric computation, A/B comparison, and LLM-as-judge scoring.
We use **pre-recorded responses** so the notebook runs without API keys.

If you have not read [prompt-evaluation.md](./prompt-evaluation.md) yet, start there first —
it explains the concepts. This notebook shows them in code.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import json

# ── Pre-recorded responses for two prompt variants ───────────────────
# Prompt A: simple — "Classify this review as POSITIVE, NEGATIVE, or NEUTRAL."
# Prompt B: detailed — includes role, format instruction, and 3 examples

TEST_SUITE = [
    {"input": "I love this product!", "expected": "POSITIVE"},
    {"input": "Terrible experience, never again", "expected": "NEGATIVE"},
    {"input": "It's okay I guess", "expected": "NEUTRAL"},
    {"input": "Best purchase I've ever made!", "expected": "POSITIVE"},
    {"input": "Waste of money", "expected": "NEGATIVE"},
    {"input": "The product arrived on time", "expected": "NEUTRAL"},
    {"input": "Not great, not terrible", "expected": "NEUTRAL"},
    {"input": "I would not recommend this", "expected": "NEGATIVE"},
    {"input": "Exceeded my expectations!", "expected": "POSITIVE"},
    {"input": "Meh", "expected": "NEUTRAL"},
    {"input": "Five stars!", "expected": "POSITIVE"},
    {"input": "Broke after one day", "expected": "NEGATIVE"},
    {"input": "Does what it says", "expected": "NEUTRAL"},
    {"input": "Absolutely wonderful experience", "expected": "POSITIVE"},
    {"input": "Could be better", "expected": "NEUTRAL"},
]

# Prompt A responses (simple prompt — no examples, no format instruction)
PROMPT_A_RESPONSES = [
    "Positive",           # slightly different format
    "NEGATIVE",
    "This seems positive, leaning neutral.",  # verbose, wrong
    "POSITIVE",
    "NEGATIVE",
    "POSITIVE",           # wrong — neutral item classified positive
    "Mixed",              # wrong format — not in our label set
    "NEGATIVE",
    "This is a positive review.",  # verbose format
    "NEUTRAL",
    "Positive!!!",        # close but format is off
    "NEGATIVE",
    "NEUTRAL",
    "POSITIVE",
    "Somewhat negative",  # wrong
]

# Prompt B responses (detailed prompt — role + 3 examples + format)
PROMPT_B_RESPONSES = [
    "POSITIVE",
    "NEGATIVE",
    "NEUTRAL",
    "POSITIVE",
    "NEGATIVE",
    "NEUTRAL",
    "NEUTRAL",
    "NEGATIVE",
    "POSITIVE",
    "NEUTRAL",
    "POSITIVE",
    "NEGATIVE",
    "NEUTRAL",
    "POSITIVE",
    "NEUTRAL",
]

# LLM-as-judge scores for a summarization task
JUDGE_SCORES = [
    {"response_id": 1, "accuracy": 4, "completeness": 3, "clarity": 5, "format": 5},
    {"response_id": 2, "accuracy": 5, "completeness": 4, "clarity": 4, "format": 5},
    {"response_id": 3, "accuracy": 3, "completeness": 2, "clarity": 5, "format": 4},
    {"response_id": 4, "accuracy": 5, "completeness": 5, "clarity": 5, "format": 5},
    {"response_id": 5, "accuracy": 4, "completeness": 4, "clarity": 3, "format": 5},
]

print("Setup complete.")
print(f"Test suite: {len(TEST_SUITE)} cases")
print(f"Prompt A responses: {len(PROMPT_A_RESPONSES)}")
print(f"Prompt B responses: {len(PROMPT_B_RESPONSES)}")
print(f"Judge scores: {len(JUDGE_SCORES)} responses scored")

## Part 1: Building a Test Suite

A test suite is a list of inputs with known correct answers.
We already defined ours above. Let's inspect it.

In [ ]:
# ── Inspect the test suite ──────────────────────────────────────────
from collections import Counter

label_counts = Counter(t["expected"] for t in TEST_SUITE)

print("TEST SUITE SUMMARY")
print("=" * 50)
print(f"Total test cases: {len(TEST_SUITE)}")
print(f"Label distribution:")
for label, count in label_counts.most_common():
    bar = '█' * (count * 3)
    print(f"  {label:10s}: {count} {bar}")

print()
print("A good test suite has balanced labels.")
print("If one label is missing, the AI could get 100% by always guessing the majority.")

## Part 2: Computing Metrics

We will compute four metrics for each prompt:
- **Accuracy**: how often the AI got the correct label
- **Format compliance**: how often the output was exactly one of our valid labels
- **Per-class accuracy**: accuracy broken down by POSITIVE, NEGATIVE, NEUTRAL

In [ ]:
VALID_LABELS = {"POSITIVE", "NEGATIVE", "NEUTRAL"}


def evaluate_responses(test_suite, responses):
    """Compute accuracy and format compliance for a set of responses."""
    correct = 0
    format_ok = 0
    per_class = {label: {"total": 0, "correct": 0} for label in VALID_LABELS}
    details = []
    
    for test, response in zip(test_suite, responses):
        expected = test["expected"]
        clean_response = response.strip()
        
        # Format compliance: is the response exactly one of our labels?
        is_valid_format = clean_response in VALID_LABELS
        if is_valid_format:
            format_ok += 1
        
        # Accuracy: does it match the expected label?
        is_correct = clean_response == expected
        if is_correct:
            correct += 1
        
        # Per-class tracking
        per_class[expected]["total"] += 1
        if is_correct:
            per_class[expected]["correct"] += 1
        
        details.append({
            "input": test["input"],
            "expected": expected,
            "got": clean_response,
            "correct": is_correct,
            "valid_format": is_valid_format,
        })
    
    n = len(test_suite)
    return {
        "accuracy": correct / n,
        "format_compliance": format_ok / n,
        "correct": correct,
        "total": n,
        "per_class": per_class,
        "details": details,
    }


# ── Evaluate both prompts ───────────────────────────────────────────
results_a = evaluate_responses(TEST_SUITE, PROMPT_A_RESPONSES)
results_b = evaluate_responses(TEST_SUITE, PROMPT_B_RESPONSES)

print("PROMPT A (simple):")
print(f"  Accuracy:          {results_a['accuracy']:.0%} ({results_a['correct']}/{results_a['total']})")
print(f"  Format compliance: {results_a['format_compliance']:.0%}")

print(f"\nPROMPT B (detailed with examples):")
print(f"  Accuracy:          {results_b['accuracy']:.0%} ({results_b['correct']}/{results_b['total']})")
print(f"  Format compliance: {results_b['format_compliance']:.0%}")

In [ ]:
# ── Show per-class accuracy ─────────────────────────────────────────
print("PER-CLASS ACCURACY")
print("=" * 50)
print(f"{'Label':10s} {'Prompt A':15s} {'Prompt B':15s}")
print("-" * 40)
for label in ["POSITIVE", "NEGATIVE", "NEUTRAL"]:
    a_data = results_a["per_class"][label]
    b_data = results_b["per_class"][label]
    a_acc = a_data["correct"] / a_data["total"] if a_data["total"] > 0 else 0
    b_acc = b_data["correct"] / b_data["total"] if b_data["total"] > 0 else 0
    print(f"{label:10s} {a_acc:>6.0%} ({a_data['correct']}/{a_data['total']})   {b_acc:>6.0%} ({b_data['correct']}/{b_data['total']})")

print()
print("Prompt A struggles most with NEUTRAL cases — it tends to guess POSITIVE.")
print("Prompt B handles all categories well because the examples showed all three.")

## Part 3: A/B Comparison Visualization

A side-by-side bar chart makes the difference between prompts clear.

In [ ]:
# ── A/B comparison chart ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Chart 1: Overall metrics
metrics = ['Accuracy', 'Format\nCompliance']
a_scores = [results_a['accuracy'] * 100, results_a['format_compliance'] * 100]
b_scores = [results_b['accuracy'] * 100, results_b['format_compliance'] * 100]

x = range(len(metrics))
width = 0.35

bars_a = axes[0].bar([i - width/2 for i in x], a_scores, width, label='Prompt A (simple)', color='#e74c3c')
bars_b = axes[0].bar([i + width/2 for i in x], b_scores, width, label='Prompt B (detailed)', color='#2ecc71')

for bar, score in zip(bars_a, a_scores):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{score:.0f}%', ha='center', fontsize=11, fontweight='bold')
for bar, score in zip(bars_b, b_scores):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{score:.0f}%', ha='center', fontsize=11, fontweight='bold')

axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics)
axes[0].set_ylim(0, 115)
axes[0].set_title('Overall Metrics', fontsize=13)
axes[0].legend()
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# Chart 2: Per-class accuracy
labels_list = ['POSITIVE', 'NEGATIVE', 'NEUTRAL']
a_class = []
b_class = []
for label in labels_list:
    a_d = results_a['per_class'][label]
    b_d = results_b['per_class'][label]
    a_class.append(a_d['correct'] / a_d['total'] * 100 if a_d['total'] > 0 else 0)
    b_class.append(b_d['correct'] / b_d['total'] * 100 if b_d['total'] > 0 else 0)

x2 = range(len(labels_list))
bars_a2 = axes[1].bar([i - width/2 for i in x2], a_class, width, label='Prompt A', color='#e74c3c')
bars_b2 = axes[1].bar([i + width/2 for i in x2], b_class, width, label='Prompt B', color='#2ecc71')

for bar, score in zip(bars_a2, a_class):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{score:.0f}%', ha='center', fontsize=11, fontweight='bold')
for bar, score in zip(bars_b2, b_class):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{score:.0f}%', ha='center', fontsize=11, fontweight='bold')

axes[1].set_xticks(x2)
axes[1].set_xticklabels(labels_list)
axes[1].set_ylim(0, 115)
axes[1].set_title('Per-Class Accuracy', fontsize=13)
axes[1].legend()
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print("Prompt B wins on every metric. The few-shot examples make the difference.")

## Part 4: Error Analysis

Numbers tell you *how much* a prompt fails. Error analysis tells you *why* it fails.
Let's look at every case where Prompt A got it wrong.

In [ ]:
# ── Show all failures for Prompt A ──────────────────────────────────
print("PROMPT A — FAILURE ANALYSIS")
print("=" * 70)
failure_count = 0
for detail in results_a["details"]:
    if not detail["correct"]:
        failure_count += 1
        format_note = "" if detail["valid_format"] else " [FORMAT ERROR]"
        print(f"  Input:    {detail['input']}")
        print(f"  Expected: {detail['expected']}")
        print(f"  Got:      {detail['got']}{format_note}")
        print()

print(f"Total failures: {failure_count}/{results_a['total']}")
print()
print("Patterns in the failures:")
print("  1. Format errors — AI returns verbose text instead of a single label")
print("  2. NEUTRAL confusion — AI classifies neutral reviews as POSITIVE")
print("  3. Invented labels — AI uses 'Mixed' which is not in our label set")
print()
print("Fix: add examples that show the exact format and include NEUTRAL cases.")
print("This is exactly what Prompt B does — and it gets 100% accuracy.")

## Part 5: LLM-as-a-Judge

For open-ended tasks (summaries, essays), there is no single "correct" answer.
LLM-as-a-judge uses a second AI to score responses on specific criteria.

Here we use pre-recorded judge scores for 5 summary responses.

In [ ]:
# ── Show judge scores ───────────────────────────────────────────────
print("LLM-AS-JUDGE SCORES (1-5 scale)")
print("=" * 60)
print(f"{'Response':10s} {'Accuracy':10s} {'Complete':10s} {'Clarity':10s} {'Format':10s} {'Average':10s}")
print("-" * 60)

all_averages = []
for score in JUDGE_SCORES:
    avg = (score["accuracy"] + score["completeness"] + score["clarity"] + score["format"]) / 4
    all_averages.append(avg)
    print(f"  #{score['response_id']:<8d} {score['accuracy']:<10d} {score['completeness']:<10d} "
          f"{score['clarity']:<10d} {score['format']:<10d} {avg:<10.2f}")

overall_avg = sum(all_averages) / len(all_averages)
print("-" * 60)
print(f"  {'Overall':10s} {'':10s} {'':10s} {'':10s} {'':10s} {overall_avg:<10.2f}")
print()
print(f"Average quality score: {overall_avg:.2f}/5.00")
print(f"Weakest area: completeness (some summaries miss key points)")

In [ ]:
# ── Visualize judge scores ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))

criteria = ['Accuracy', 'Completeness', 'Clarity', 'Format']
avg_scores = [
    sum(s["accuracy"] for s in JUDGE_SCORES) / len(JUDGE_SCORES),
    sum(s["completeness"] for s in JUDGE_SCORES) / len(JUDGE_SCORES),
    sum(s["clarity"] for s in JUDGE_SCORES) / len(JUDGE_SCORES),
    sum(s["format"] for s in JUDGE_SCORES) / len(JUDGE_SCORES),
]

colors = ['#3498db' if s >= 4.0 else '#e74c3c' for s in avg_scores]
bars = ax.barh(criteria, avg_scores, color=colors, edgecolor='white', linewidth=2)

for bar, score in zip(bars, avg_scores):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            f'{score:.1f}/5', va='center', fontsize=12, fontweight='bold')

ax.set_xlim(0, 5.5)
ax.set_title('Average Judge Scores by Criterion', fontsize=14)
ax.axvline(x=4.0, color='gray', linestyle='--', alpha=0.5, label='Target (4.0)')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print("Bars in blue meet the target (4.0+). Red bars need improvement.")
print("Action: improve completeness by adding 'cover all key points' to the prompt.")

## Part 6: Tracking Progress Over Versions

As you iterate on a prompt, track your scores over time to see improvement.

In [ ]:
# ── Simulated version history ───────────────────────────────────────
version_history = [
    {"version": "v1", "accuracy": 0.60, "format": 0.70, "note": "Baseline — simple prompt"},
    {"version": "v2", "accuracy": 0.75, "format": 0.90, "note": "Added format instruction"},
    {"version": "v3", "accuracy": 0.85, "format": 0.95, "note": "Added 3 examples (few-shot)"},
    {"version": "v4", "accuracy": 0.93, "format": 1.00, "note": "Added role + balanced examples"},
]

print("PROMPT VERSION HISTORY")
print("=" * 65)
print(f"{'Version':10s} {'Accuracy':12s} {'Format':12s} {'Change':30s}")
print("-" * 65)
for v in version_history:
    print(f"{v['version']:10s} {v['accuracy']:>8.0%}     {v['format']:>8.0%}     {v['note']}")

# ── Line chart ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))

versions = [v["version"] for v in version_history]
acc_values = [v["accuracy"] * 100 for v in version_history]
fmt_values = [v["format"] * 100 for v in version_history]

ax.plot(versions, acc_values, 'o-', color='#2ecc71', linewidth=2, markersize=8, label='Accuracy')
ax.plot(versions, fmt_values, 's-', color='#3498db', linewidth=2, markersize=8, label='Format Compliance')

for i, (v, a, f) in enumerate(zip(versions, acc_values, fmt_values)):
    ax.annotate(f'{a:.0f}%', (v, a), textcoords="offset points", xytext=(0, 10), ha='center', fontsize=10)

ax.set_ylim(50, 105)
ax.set_title('Prompt Quality Over Iterations', fontsize=14)
ax.set_ylabel('Score (%)', fontsize=12)
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print("Each iteration improved the prompt. The biggest jump came from")
print("adding few-shot examples (v2 → v3). This matches what we learned")
print("in the few-shot learning notebook.")

## Summary

What you just built:

1. **Test suite**: a set of inputs with expected outputs for systematic evaluation
2. **Metrics**: accuracy, format compliance, and per-class breakdown
3. **A/B comparison**: side-by-side evaluation of two prompt variants
4. **Error analysis**: identifying patterns in failures to guide improvements
5. **LLM-as-judge**: using AI to score open-ended responses on specific criteria
6. **Version tracking**: monitoring quality over prompt iterations

For the concepts behind evaluation, read [prompt-evaluation.md](./prompt-evaluation.md).
This is the final notebook in the prompt engineering module.

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)